# **Fitur Unik 1: Penanganan Ketidakseimbangan Kelas (Focal Loss dan Threshold Optimal)**

Notebook 04 menunjukkan bahwa model tanpa fitur unik cukup baik dalam mengurutkan risiko (ROC-AUC 0,819), tetapi dengan threshold bawaan 0,5 model hanya menemukan **11,42%** responden diabetes di data validation. Fitur unik pertama dirancang untuk mengatasi masalah ini lewat dua komponen:

1. **Komponen loss (saat training):** mengganti binary cross-entropy dengan **Focal Loss**, supaya model lebih fokus pada sampel yang sulit, yang kebanyakan adalah kasus diabetes.
2. **Komponen keputusan (setelah training):** mengganti threshold 0,5 dengan **threshold yang dipilih di data validation** berdasarkan kriteria yang sesuai untuk skrining.

Dampak tiap komponen diukur terpisah lewat desain **ablation 2 x 2** (loss BCE atau Focal, dikali threshold 0,5 atau threshold optimal). Dengan begitu bisa dilihat komponen mana yang benar-benar memberi perbaikan.

Arsitektur, learning rate, dan batch size diambil dari `04_konfigurasi_final.json` tanpa perubahan. Semua pemilihan dilakukan di data validation. **Data test tidak disentuh** sampai notebook 07.

**Penanggung jawab:** Johannes Hartanto (231110730)

## **1. Import Library dan Konfigurasi**

Eksperimen di notebook ini melatih 16 model (5 variasi loss x 3 seed, ditambah 1 model final) dan butuh sekitar 20 sampai 30 menit di CPU. Hasil BCE dari notebook sebelumnya dipakai ulang. Saklar `RUN_EXPERIMENTS = False` memuat hasil yang tersimpan tanpa training ulang.

In [2]:
import sys
import json
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedKFold

from src import config
from src import preprocessing as prep
from src import model as M
from src import evaluation as ev
from src import visualization as viz

RUN_EXPERIMENTS = True

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
viz.set_style()
WARNA_KELAS = {0: "#4C72B0", 1: "#DD8452"}
WARNA_METRIK = {"recall": "#2a78d6", "precision": "#eb6834", "f2": "#1baf7a"}

MODEL_BCE_PATH = config.MODELS_DIR / "mlp_tanpa_fitur_unik.keras"
MODEL_FINAL_PATH = config.MODELS_DIR / "mlp_final.keras"
MODEL_CONFIG_PATH = config.MODELS_DIR / "model_config.json"
PROB_PATH = config.RESULTS_DIR / "05_prob_validation_seed42.npz"

print("TensorFlow :", tf.__version__)
print("RUN_EXPERIMENTS :", RUN_EXPERIMENTS)

TensorFlow : 2.21.0
RUN_EXPERIMENTS : True


## **2. Memuat Data, Konfigurasi, dan Model Tanpa Fitur Unik**

In [3]:
data = prep.prepare_datasets()
assert data["test_fingerprint"] == prep.load_preprocessing_config()["split"]["test_fingerprint"]
y_val = data["y_val"].astype(int)

konfig_04 = M.load_json(config.RESULTS_DIR / "04_konfigurasi_final.json")
cfg_dasar = konfig_04["konfigurasi"]
model_bce = keras.models.load_model(MODEL_BCE_PATH)

n0, n1 = int((data["y_train"] == 0).sum()), int((data["y_train"] == 1).sum())
alpha_seimbang = round(n0 / (n0 + n1), 3)

print("Konfigurasi dasar   :", cfg_dasar)
print(f"Model tanpa fitur unik: {MODEL_BCE_PATH.relative_to(ROOT)} (val PR-AUC {konfig_04['model_final_val_pr_auc']:.4f})")
print(f"Train kelas 0 / 1   : {n0:,} / {n1:,}")
print(f"Alpha seimbang      : {alpha_seimbang} (proporsi kelas 0 di data train)")

Konfigurasi dasar   : {'hidden_units': [128, 64], 'dropout': 0.2, 'l2': 0.0, 'batch_norm': False, 'learning_rate': 0.001, 'batch_size': 512}
Model tanpa fitur unik: models\mlp_tanpa_fitur_unik.keras (val PR-AUC 0.4463)
Train kelas 0 / 1   : 135,700 / 24,511
Alpha seimbang      : 0.847 (proporsi kelas 0 di data train)
